In [1]:
!pip install -q sentence-transformers transformers torch

In [2]:
##  Bi-Encoder Dense Retrieval
## Dense retrieval converts documents and queries into vectors to find top matches rapidly.
from sentence_transformers import SentenceTransformer, util

# 1. Load Bi-Encoder Model
bi_encoder = SentenceTransformer("all-MiniLM-L6-v2")

# 2. Knowledge Base (Documents)
documents = [
    "The Transformer model was introduced in 2017 in the paper 'Attention Is All You Need'.",
    "BERT is an encoder-only model designed for natural language understanding and representation.",
    "GPT-4 is a decoder-only generative autoregressive language model created by OpenAI.",
    "Retrieval-Augmented Generation (RAG) grounds LLM outputs using retrieved context."
]

# 3. Embed Documents
doc_embeddings = bi_encoder.encode(documents, convert_to_tensor=True)

# 4. Process Query
query = "Which model paper introduced transformers?"
query_embedding = bi_encoder.encode(query, convert_to_tensor=True)

# 5. Compute Cosine Similarities & Get Top Hits
hits = util.semantic_search(query_embedding, doc_embeddings, top_k=2)[0]

print(f"Query: '{query}'\n")
print("--- DENSE RETRIEVAL HITS ---")
for hit in hits:
    doc_id = hit['corpus_id']
    score = hit['score']
    print(f"Score: {score:.4f} | Document: {documents[doc_id]}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Query: 'Which model paper introduced transformers?'

--- DENSE RETRIEVAL HITS ---
Score: 0.6514 | Document: The Transformer model was introduced in 2017 in the paper 'Attention Is All You Need'.
Score: 0.2032 | Document: GPT-4 is a decoder-only generative autoregressive language model created by OpenAI.


In [3]:
## Cross-Encoder Reranking
## A Bi-Encoder computes vectors independently. A Cross-Encoder feeds (Query, Document) together into full attention layers for superior ranking precision.

from sentence_transformers import CrossEncoder

# Load a dedicated Cross-Encoder reranker model
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Create query-document pairs from candidates
pairs = [[query, doc] for doc in documents]

# Calculate relevance scores
scores = cross_encoder.predict(pairs)

# Combine and sort results
ranked_results = sorted(zip(scores, documents), key=lambda x: x[0], reverse=True)

print("--- CROSS-ENCODER RERANKED RESULTS ---")
for score, doc in ranked_results:
    print(f"Relevance Score: {score:.4f} | Document: {doc}")


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

--- CROSS-ENCODER RERANKED RESULTS ---
Relevance Score: 6.3618 | Document: The Transformer model was introduced in 2017 in the paper 'Attention Is All You Need'.
Relevance Score: -9.6278 | Document: GPT-4 is a decoder-only generative autoregressive language model created by OpenAI.
Relevance Score: -10.4373 | Document: BERT is an encoder-only model designed for natural language understanding and representation.
Relevance Score: -11.4503 | Document: Retrieval-Augmented Generation (RAG) grounds LLM outputs using retrieved context.


In [9]:
## Complete RAG Pipeline
## Now, we take the top retrieved document as dynamic context and pass it to a generative model (google/flan-t5-base) to answer the question accurately.

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load a generative model for synthesis
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
rag_generator = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Extract top context from reranker
top_retrieved_context = ranked_results[0][1]

# Construct RAG Prompt
rag_prompt = f"""Answer the question based strictly on the context below.

Context:
{top_retrieved_context}

Question:
{query}

Answer:"""

# Encode the prompt
inputs = tokenizer(rag_prompt, return_tensors="pt", max_length=512, truncation=True)

# Generate the response
output_tokens = rag_generator.generate(
    **inputs,
    max_new_tokens=50,
    num_beams=4,
    early_stopping=True
)

# Decode the generated text
generated_text = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

print("--- RAG GENERATED RESPONSE ---")
print("Retrieved Context:", top_retrieved_context)
print("Generated Answer:", generated_text)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


--- RAG GENERATED RESPONSE ---
Retrieved Context: The Transformer model was introduced in 2017 in the paper 'Attention Is All You Need'.
Generated Answer: Attention Is All You Need
